# 🔬 CONSOLE LABORATOIRE INTERACTIVE — COPILOT SOMMELIER & ACCORDS VINS
> **Agent ID :** `copilot-sommelier-caviste` (v1.0.0)  
> **Rôle :** Conseiller caviste expert guidant les clients dans le choix d'un vin selon leurs plats et leur budget.  
> **Objectif :** Banc d'essai interactif pour tester le cycle de vie complet d'un tour de dialogue, le Context Engineering S0-S6, le moteur métier déterministe et la persistance SQLite WAL.


In [ ]:
import sys
import os
import json
from pathlib import Path
from IPython.display import display, HTML

current_dir = Path(os.getcwd())
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
s_root = str(project_root.resolve())
if os.name == 'nt' and not s_root.startswith('\\\\?\\'):
    s_ext = '\\\\?\\' + s_root
else:
    s_ext = s_root

for p in [str(project_root), s_root, s_ext]:
    if p not in sys.path:
        sys.path.insert(0, p)

from copilot_sommelier_caviste.domain.state_form import UniversalStateForm
from copilot_sommelier_caviste.domain.engine import ConseilCavisteEngine
from copilot_sommelier_caviste.application.context_pipeline_builder import ContextPipelineBuilder
from copilot_sommelier_caviste.adapters.persistence.sqlite_cache_adapter import SQLiteCacheAdapter

print(f"✅ Environnement initialisé avec succès pour 'Copilot Sommelier & Accords Vins'.")


---
## 📋 ÉTAPE 1 : LE FORMULAIRE VIVANT & GESTION DES SLOTS (DDD)
Initialisation de `UniversalStateForm` avec les slots déclaratifs du métier et calcul du taux de complétion initial.


In [ ]:
form = UniversalStateForm.create_default()
print(f"Total slots déclarés : {len(form.slots_definitions)}")
print(f"Complétude initiale   : {form.get_completion_percentage()} %")
print(f"Statut FSM initial    : is_complete = {form.is_complete()}")
print(f"Première question     : '{form.get_next_question()}'")


---
## 💬 ÉTAPE 2 : SIMULATION D'UN TOUR DE DIALOGUE & EXTRACTION DES FAITS
Mise à jour des faits confirmés suite à une interaction utilisateur et recalcul automatique de la FSM.


In [ ]:
form.update_fact("type_plat", "REF-001")

print(f"Faits confirmés         : {form.confirmed_facts}")
print(f"Nouveau taux complétude : {form.get_completion_percentage()} %")
print(f"Prochaine question      : '{form.get_next_question()}'")


---
## 🧠 ÉTAPE 3 : CONSTRUCTEUR DE CONTEXTE DÉTERMINISTE (SECTIONS S0 À S6)
Génération du prompt système injecté au LLM garantissant le respect de la doctrine et zéro hallucination.


In [ ]:
builder = ContextPipelineBuilder()
prompt = builder.build_system_prompt(
    session_id="SES-LAB-2026-X1",
    confirmed_facts=form.confirmed_facts,
    missing_slots=[s.name for s in form.get_missing_slots()],
    next_question=form.get_next_question() or "Aucune"
)

print(prompt[:600] + "\n... [Tronqué pour affichage] ...")


---
## ⚙️ ÉTAPE 4 : EXÉCUTION DU MOTEUR MÉTIER DÉTERMINISTE (ZÉRO IA)
Exécution de l'action métier sécurisée (vérification des invariants et émission du statut).


In [ ]:
engine = ConseilCavisteEngine()
exec_result = engine.execute("SES-LAB-2026-X1", form.confirmed_facts)

print(f"Succès exécution : {exec_result.success}")
print(f"Statut métier    : {exec_result.status_code}")
print(f"Message          : {exec_result.message}")
print(f"Payload données  : {exec_result.data}")


---
## 💾 ÉTAPE 5 : PERSISTANCE HYBRIDE DANS LE CACHE LOCAL (SQLITE WAL < 2ms)
Sauvegarde atomique de la session et rechargement vérifié.


In [ ]:
cache = SQLiteCacheAdapter(":memory:")
cache.save_session("SES-LAB-2026-X1", {
    "agent_id": "copilot-sommelier-caviste",
    "completion_rate": form.get_completion_percentage(),
    "facts": form.confirmed_facts
})

reloaded = cache.load_session("SES-LAB-2026-X1")
assert reloaded is not None
print(f"✅ Session rechargée avec succès depuis SQLite : {reloaded}")


---
## 🎨 ÉTAPE 6 : RENDU VISUEL INTERACTIF HTML / CSS DE LA TABLETTE GPS
Projection visuelle de la carte copilote telle qu'elle apparaîtra dans le cockpit de l'utilisateur.


In [ ]:
html_card = f'''
<div style="font-family: system-ui, sans-serif; border: 1px solid #10b981; border-radius: 12px; padding: 20px; max-width: 500px; background: #f0fdf4; box-shadow: 0 4px 12px rgba(0,0,0,0.05);">
  <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px;">
    <h3 style="margin: 0; color: #065f46; font-size: 18px;">🤖 Copilot Sommelier & Accords Vins</h3>
    <span style="background: #10b981; color: white; padding: 4px 10px; border-radius: 999px; font-size: 12px; font-weight: bold;">FSM ACTIVE</span>
  </div>
  <p style="margin: 0 0 12px 0; color: #047857; font-size: 14px;"><strong>Rôle :</strong> Conseiller caviste expert guidant les clients dans le choix d'un vin selon leurs plats et leur budget.</p>
  <div style="background: white; border-radius: 8px; padding: 12px; border: 1px solid #d1fae5; margin-bottom: 12px;">
    <div style="font-size: 12px; color: #6b7280; margin-bottom: 4px;">Taux de Complétion :</div>
    <div style="background: #e5e7eb; border-radius: 999px; height: 10px; overflow: hidden;">
      <div style="background: #10b981; width: {form.get_completion_percentage()}%; height: 100%;"></div>
    </div>
    <div style="text-align: right; font-size: 12px; font-weight: bold; color: #065f46; margin-top: 4px;">{form.get_completion_percentage()} %</div>
  </div>
  <div style="font-size: 13px; color: #374151;">
    <strong>Faits confirmés :</strong> {json.dumps(form.confirmed_facts)}
  </div>
</div>
'''
display(HTML(html_card))
